In [ ]:
import torch
import fm

if torch.cuda.is_available():
    device = 'cuda'
else:
    device = 'cpu'

print(f'using {device} device')

# 1. Load RNA-FM model
model, alphabet = fm.pretrained.rna_fm_t12()
batch_converter = alphabet.get_batch_converter()

model.to(device)

model.eval()  # disables dropout for deterministic results



: 

In [6]:
# 2. Prepare data
data = [
    ("RNA1", "GGGUGCGAUCAUACCAGCACUAAUGCCCUCCUGGGAAGUCCUCGUGUUGCACCCCU"),
    ("RNA2", "GGGUGUCGCUCAGUUGGUAGAGUGCUUGCCUGGCAUGCAAGAAACCUUGGUUCAAUCCCCAGCACUGCA"),
    ("RNA3", "CGAUUCNCGUUCCC--CCGCCUCCA"),
]
batch_labels, batch_strs, batch_tokens = batch_converter(data)

# 3. Extract embeddings (on device)
with torch.no_grad():
    results = model(batch_tokens.to(device), repr_layers=[12])
token_embeddings = results["representations"][12]


print(token_embeddings.shape)

torch.Size([3, 71, 640])


In [ ]:
import torch
import fm
import pandas as pd
import numpy as np
import pickle
from pathlib import Path

# ── 配置 ──────────────────────────────────────────
CSV_PATH = "final_ncRNA_MetS.csv"   # 修改为你的文件路径
OUTPUT_CSV = "ncRNA_embeddings.csv"
OUTPUT_PKL = "ncRNA_embeddings.pkl"
MAX_LEN = 1022
BATCH_SIZE = 4               # 显存不足时可以调小
REPR_LAYER = 12
# ──────────────────────────────────────────────────

# 1. 加载模型
model, alphabet = fm.pretrained.rna_fm_t12()
batch_converter = alphabet.get_batch_converter()
model.eval()

# 2. 读取 CSV
df = pd.read_csv(CSV_PATH)

def split_sequence(seq, max_len=MAX_LEN):
    """将序列切分为不重叠的等长片段"""
    return [seq[i:i+max_len] for i in range(0, len(seq), max_len)]

def get_sequence_embedding(seq_id, seq):
    """对单条序列分段取 embedding，返回各段均值的平均"""
    segments = split_sequence(seq)
    data = [(f"{seq_id}_seg{i}", seg) for i, seg in enumerate(segments)]
    
    seg_embeddings = []
    # 分 batch 处理各段
    for i in range(0, len(data), BATCH_SIZE):
        batch = data[i:i+BATCH_SIZE]
        _, _, batch_tokens = batch_converter(batch)
        with torch.no_grad():
            results = model(batch_tokens, repr_layers=[REPR_LAYER])
        token_repr = results["representations"][REPR_LAYER]  # (B, L, D)
        # 每段：去掉首尾特殊 token，对剩余位置取均值
        for j in range(token_repr.shape[0]):
            seg_len = len(batch[j][1])
            seg_emb = token_repr[j, 1:seg_len+1, :].mean(dim=0)  # (D,)
            seg_embeddings.append(seg_emb)
    
    # 所有段的 embedding 取平均
    seq_embedding = torch.stack(seg_embeddings).mean(dim=0)  # (D,)
    return seq_embedding.cpu().numpy()

# 3. 逐条处理
results_dict = {}  # {id: embedding_array}

for _, row in df.iterrows():
    seq_id = row["Biomoleculename"]
    seq = str(row["sequence"]).strip()
    print(f"Processing {seq_id}, length={len(seq)} ...")
    emb = get_sequence_embedding(seq_id, seq)
    results_dict[seq_id] = emb

# 4. 保存为 CSV
emb_dim = len(next(iter(results_dict.values())))
emb_df = pd.DataFrame(
    {seq_id: emb for seq_id, emb in results_dict.items()}
).T  # shape: (n_sequences, emb_dim)


emb_df.index.name = "Biomoleculename"
emb_df.columns = [f"dim_{i}" for i in range(emb_dim)]
emb_df.to_csv(OUTPUT_CSV)
print(f"Saved CSV -> {OUTPUT_CSV}")

# 5. 保存为 PKL
with open(OUTPUT_PKL, "wb") as f:
    pickle.dump(results_dict, f)
print(f"Saved PKL -> {OUTPUT_PKL}")

Processing AC092159.2, length=1943 ...
Processing AK098656, length=1512 ...
Processing ASMER-2, length=472 ...
Processing CARMEN, length=2833 ...
Processing CDKN2B-AS1, length=7173 ...
Processing CTBP1-AS2, length=7575 ...
Processing DNM3OS, length=2538 ...
Processing ENST00000550337.1, length=1960 ...
Processing FENDRR, length=3705 ...
Processing GAS5, length=3145 ...
Processing GYG2P1, length=149 ...
Processing H19, length=2396 ...
Processing H19, length=2396 ...
Processing HCG27_201, length=2018 ...
Processing hcmv-miR-UL112, length=22 ...
Processing HI-LNC25, length=2591 ...
Processing HOTAIR, length=2415 ...
Processing HOTAIR, length=2415 ...
Processing HOXA11-AS1, length=1739 ...
Processing hsa-let-7, length=22 ...
Processing hsa-let-7a, length=21 ...
Processing hsa-let-7b, length=22 ...
Processing hsa-let-7c, length=22 ...
Processing hsa-let-7d, length=22 ...
Processing hsa-let-7e, length=22 ...
Processing hsa-let-7e, length=22 ...
Processing hsa-let-7e, length=22 ...
Processing

In [10]:
with open("lncRNA_embeddings.pkl", "rb") as f:
    data = pickle.load(f)
    print(data)
# data 是 dict：{id(int/str): numpy array (640,)}

{1: array([-4.01832074e-01,  4.81379390e-01, -2.88978249e-01, -3.27887893e-01,
       -3.40736538e-01,  9.94598269e-02, -1.94899440e-02,  9.41427410e-01,
       -2.42112949e-01, -6.09315336e-01, -1.99488044e-01, -1.69700906e-01,
        6.34720445e-01,  3.43164533e-01, -5.77252567e-01,  3.23680520e-01,
       -3.54543328e-01,  3.94325018e-01,  1.75382897e-01, -1.95598483e-01,
        5.42480528e-01, -1.49608955e-01,  1.19270146e-01,  3.34667653e-01,
        5.18197455e-02, -4.16036934e-01, -5.78483164e-01, -2.95866370e-01,
       -7.96997547e-01,  2.34192297e-01,  2.42128626e-01, -3.28456849e-01,
       -6.31934583e-01,  6.68830395e-01, -5.57290077e-01, -3.35193537e-02,
       -3.77297163e-01, -5.52216947e-01, -1.64983094e-01,  4.69148308e-01,
        8.26047286e-02,  3.72601181e-01,  3.20097446e-01,  3.58713716e-01,
        5.74903071e-01, -2.93667972e-01, -2.16341153e-01,  1.03654183e-01,
        2.82950044e-01,  1.35547698e-01,  1.00129008e-01, -3.23219836e-01,
       -6.37036741e-0